# Lab 3 — Running a Single-Shot Detector, and Measuring Where It Fails

**Unit.** `fast_object_detection_networks` · **Duration.** 1 hour

A pretrained detector, 32 real images, and their ground-truth boxes. Nothing is trained
here: the lab is about what a trained single-shot detector gets right and what it gets
wrong, measured on numbers you produce yourself.

1. Run the detector and read what one forward pass returns.
2. Match its boxes to the ground truth by **IoU**, draw the two against each other, and
   compute the **localisation error** and the **label accuracy**.
3. Find the two places it fails: the confidence threshold, and small objects.

The unit claims that a single-shot detector localises loosely and classifies well, and
that small objects are its structural weakness. By the end of Exercise 3 you will have
measured both.

Work through the cells in order. Each cell marked **Your turn** is followed by a CHECK
cell that prints `[ok]` when you have it right.

## Setup

Provided. Downloads COCO128 (7 MB) and the YOLOv3-tiny weights (23 MB), both cached, then
draws four images with their ground-truth boxes so you can see what the annotations are.

On Colab the first cell installs `ultralytics`; running locally it does nothing.

In [ ]:
# COLAB SETUP -- run this cell first.  Outside Colab it does nothing.
import importlib.util
import subprocess
import sys
import urllib.request

try:  # find_spec raises when the parent package is absent, so import instead
    import google.colab  # noqa: F401

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # Colab already ships torch and torchvision; install only what is missing.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "lightning", "torchmetrics", "ultralytics==8.3.228"], check=True)

# Fetch the lab's support module when it is not already beside the notebook.
if importlib.util.find_spec("lab3_utils") is None:
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/kenpu-uoit/csci4052u-machine-learning-2/main/labs/lab3/lab3_utils.py", "lab3_utils.py")
    print("downloaded lab3_utils.py")


In [ ]:
import time

import matplotlib.pyplot as plt
import torch

from lab3_utils import (
    CONF, N_IMAGES, SMALL_AREA,
    image_paths, image_size, load_ground_truth, load_model,
    draw_boxes, show_comparison, sweep_table,
    check_gt_to_xyxy, check_predictions, check_iou, check_match, check_figure,
    check_metrics, check_sweep, check_sweep_figure, check_scale,
)

torch.manual_seed(0)

IMAGES = image_paths(N_IMAGES)
model = load_model()
NAMES = model.names

print(f"{len(IMAGES)} images, {len(NAMES)} COCO classes, confidence threshold {CONF}")
print(f"first image: {IMAGES[0].name}, {image_size(IMAGES[0])[0]}x{image_size(IMAGES[0])[1]} px")
print(f"its annotation rows (class, cx, cy, w, h), normalised to [0, 1]:")
for row in load_ground_truth(IMAGES[0])[:3]:
    print("   ", NAMES[row[0]], tuple(round(v, 3) for v in row[1:]))

In [ ]:
# What the ground truth looks like, drawn with the helper you will reuse in 2b.
fig, axes = plt.subplots(1, 4, figsize=(14, 3.2))
for ax, path in zip(axes, IMAGES[:4]):
    w, h = image_size(path)
    boxes = [(c, (cx - bw / 2) * w, (cy - bh / 2) * h, (cx + bw / 2) * w, (cy + bh / 2) * h)
             for c, cx, cy, bw, bh in load_ground_truth(path)]
    draw_boxes(ax, path, boxes, "#0d6efd", NAMES)
fig.suptitle("COCO128 ground truth", fontsize=11)
fig.tight_layout()
plt.show()

---

## Exercise 1 — One forward pass, and what comes out of it

*Outcome: formulate detection as one regression from image pixels to boxes and class
probabilities.*

### 1a · Ground truth into pixel corners — **Your turn**

COCO128 stores each object as `class cx cy w h`, where the four box numbers are the
**centre** and the **width and height**, each normalised to $[0, 1]$ against the image.
Everything downstream wants **corners in pixels**.

Write `gt_to_xyxy(rows, width, height)` returning a list of
`(class, x1, y1, x2, y2)`, keeping the class index as it is.

In [ ]:
def gt_to_xyxy(rows, width, height):
    """YOLO-format rows -> (class, x1, y1, x2, y2) in pixels."""
    # YOUR CODE HERE
    raise NotImplementedError()

In [ ]:
check_gt_to_xyxy(gt_to_xyxy, IMAGES)

GROUND_TRUTH = [gt_to_xyxy(load_ground_truth(p), *image_size(p)) for p in IMAGES]

### 1b · Run the detector — **Your turn**

Call the model on all 32 images at once and collect what it returns.

```python
results = model.predict([str(p) for p in IMAGES], device="cpu", conf=CONF, verbose=False)
```

Each `result` carries `result.boxes` with three parallel tensors: `.cls`, `.xyxy` and
`.conf`. Build `predictions` — **one list per image**, in the same order as `IMAGES`, of
`(class, x1, y1, x2, y2, score)` tuples with plain Python numbers.

Keep an image that found nothing as an empty list rather than dropping it; the lists must
line up with `GROUND_TRUTH`.

In [ ]:
t0 = time.time()
# YOUR CODE HERE
raise NotImplementedError()
elapsed = time.time() - t0

In [ ]:
check_predictions(predictions, IMAGES, elapsed)

---

## Exercise 2 — Matching, drawing, and the error profile

*Outcome: diagnose a single-shot detector's error profile and its structural limits.*

### 2a · Intersection over union — **Your turn**

$$\mathrm{IoU}(A, B) = \frac{|A \cap B|}{|A \cup B|}$$

Write `iou(box_a, box_b)` taking two `(x1, y1, x2, y2)` tuples. Two boxes that do not
touch must give exactly `0.0`, which means clamping each overlapping side at zero
**before** multiplying them.

In [ ]:
def iou(box_a, box_b):
    """Intersection over union of two (x1, y1, x2, y2) boxes."""
    # YOUR CODE HERE
    raise NotImplementedError()

In [ ]:
check_iou(iou)

### 2b · Match, then look — **Your turn**

Two parts.

**The matcher.** Write `match(gts, preds, thr=0.5)` returning `(gt_index, pred_index, iou)`
triples. Walk the ground-truth boxes in order; each takes the **unused** prediction of
highest IoU with it, and the pair is kept only when that IoU reaches `thr`. A prediction
that has been taken is struck off — otherwise two overlapping objects both claim the same
box and recall is overstated.

The prediction tuples carry a score in position 5; `iou` wants corners, which are
positions 1 to 4 of both.

**The picture.** Then call
`show_comparison(IMAGES[:4], GROUND_TRUTH[:4], predictions[:4], NAMES)` and keep the
figure it returns as `fig`. Look at it before moving on: that is what an IoU of 0.8 and a
recall of under a half actually look like.

In [ ]:
def match(gts, preds, thr=0.5):
    """Greedy IoU matching -> list of (gt_index, pred_index, iou)."""
    # YOUR CODE HERE
    raise NotImplementedError()


fig = None  # replace with the figure from show_comparison

In [ ]:
check_match(match, GROUND_TRUTH, predictions)
check_figure(fig)

### 2c · Three numbers — **Your turn**

Over all 32 images, compute:

- **`recall`** — matched boxes divided by the number of ground-truth boxes.
- **`mean_iou`** — the mean IoU over matched pairs only. This is the localisation error:
  a perfect box scores 1.0.
- **`label_acc`** — of the matched pairs, the fraction whose predicted class equals the
  ground-truth class.

A matched pair is `(gi, pi, v)` from `match`; its ground-truth class is
`GROUND_TRUTH[image][gi][0]` and its predicted class is `predictions[image][pi][0]`.

In [ ]:
# YOUR CODE HERE
raise NotImplementedError()

print(f"recall         {recall:.1%}")
print(f"mean IoU       {mean_iou:.3f}")
print(f"label accuracy {label_acc:.1%}")

In [ ]:
check_metrics(recall, mean_iou, label_acc)

---

## Exercise 3 — Where it fails

*Outcome: choose a detector for a deployment constraint by reasoning about speed,
accuracy and object scale.*

### 3a · The confidence threshold — **Your turn**

Re-run the detector at each threshold in `THRESHOLDS` and recompute the three numbers.
Build `rows` as `(conf, n_predictions, recall, mean_iou, label_acc)`, one tuple per
threshold, in increasing order.

Then plot **recall** and **mean IoU** against the threshold on twin axes
(`ax2 = ax.twinx()`), and keep the figure as `sweep_fig`.

Four more passes over 32 images; about four seconds.

In [ ]:
THRESHOLDS = [0.05, 0.10, 0.25, 0.50]

# YOUR CODE HERE
raise NotImplementedError()

sweep_table(rows)

In [ ]:
check_sweep(rows)
check_sweep_figure(sweep_fig)

### 3b · Small objects — **Your turn**

Split the ground-truth boxes by how much of the image they occupy. A box is **small** when
its area is below `SMALL_AREA` (5%) of the image area, and **large** otherwise.

At the original `CONF`, compute `small_recall` and `large_recall` — within each group, the
fraction of ground-truth boxes that found a match — along with the group sizes `n_small`
and `n_large`.

A ground-truth box counts as found when `match` returned a pair whose `gt_index` is its
index.

In [ ]:
# YOUR CODE HERE
raise NotImplementedError()

print(f"small (<{SMALL_AREA:.0%} of the image): {small_recall:.1%} of {n_small}")
print(f"large:                      {large_recall:.1%} of {n_large}")

In [ ]:
check_scale(small_recall, large_recall, n_small, n_large)

---

## What this lab covered, and what it did not

Measured, on your own numbers:

- **One forward pass** turns an image into every box it contains, with no proposal stage,
  at a cost that does not grow with the number of objects.
- **The error profile**: the labels are almost always right, the boxes are loose, and more
  than half the objects are missed entirely. Localisation and recall are where a
  single-shot detector loses, not classification.
- **The confidence threshold** trades recall against precision and cannot improve both.
- **Object scale** is the structural limit: small objects are found at a fraction of the
  rate of large ones, because the map predictions are read from is too coarse to carry
  them.

Out of scope here, and covered in the lecture notes:

- training a detector, the YOLO loss, and the responsible-predictor rule;
- SSD's default boxes, its matching rule, and prediction layers at several resolutions —
  the answer to the small-object result in 3b;
- hard negative mining and the augmentation that makes single-shot training work;
- YOLOv2's dimension clusters and direct location prediction;
- mean average precision, the metric the field actually reports. Recall at a single IoU
  threshold is a deliberate simplification: it is one point on the curve mAP integrates.

The model here is `yolov3-tinyu.pt` — YOLOv3-tiny as Ultralytics publishes it, with an
anchor-free head retrofitted. It stands in for the early single-shot detectors of the
unit; no maintained PyTorch port of YOLOv2 with pretrained weights exists.